In [1]:
import pandas as pd

file_path = "../sandhi_data.xlsx"

df = pd.read_excel(file_path)

print(df.head())
print(df.columns)
print("Total samples:", len(df))

                                Word                               Split
0                        प्रथमोऽङ्कः                        प्रथमः+अङ्कः
1                            शब्द इव                            शब्दः+इव
2                             इत इतः                             इतः+इतः
3                            कुतो नु                             कुतः+नु
4  खल्वेष समुत्थितो समुत्थितो ध्वनिः  खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
Index(['Word', 'Split'], dtype='str')
Total samples: 13930


In [2]:
import unicodedata

def clean_text(text):
    text = str(text).strip()
    text = unicodedata.normalize("NFC", text)
    text = " ".join(text.split())
    return text

def preprocess_pair(inp, out):
    inp = clean_text(inp)
    out = clean_text(out)
    
    # Convert + to space
    out = out.replace("+", " ")
    
    # Add T5 task prefix
    inp = "split: " + inp
    
    return inp, out

data = []
for _, row in df.iterrows():
    inp, out = preprocess_pair(row['Word'], row['Split'])
    data.append((inp, out))

print("Processed samples:", len(data))


Processed samples: 13930


In [3]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(data, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print("Training samples: {}, Validation samples: {}, Test samples: {}".format(len(train_data), len(val_data), len(test_data)))


Training samples: 11144, Validation samples: 1393, Test samples: 1393


In [4]:
from transformers import AutoTokenizer, T5ForConditionalGeneration

model_name = "google/byt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

model.gradient_checkpointing_enable()
model.config.use_cache = False

/home/viserion/0_FYP/Phase_2/models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/viserion/0_FYP/Phase_2/models/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [5]:
import torch
from torch.utils.data import Dataset

class SandhiDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=64):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        inp, out = self.data[idx]

        inputs = self.tokenizer(
            inp,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            out,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        labels = targets["input_ids"]
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels.squeeze()
        }

train_dataset = SandhiDataset(train_data, tokenizer)
val_dataset = SandhiDataset(val_data, tokenizer)
test_dataset = SandhiDataset(test_data, tokenizer)


In [6]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./byt5_sandhi",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    fp16=True,
    load_best_model_at_end=True
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()


                                                  
  8%|▊         | 348/4176 [03:54<41:53,  1.52it/s]

{'eval_loss': nan, 'eval_runtime': 6.424, 'eval_samples_per_second': 216.844, 'eval_steps_per_second': 27.242, 'epoch': 1.0}


 12%|█▏        | 500/4176 [05:35<40:03,  1.53it/s]  

{'loss': 26.5367, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 1.44}


                                                  
 17%|█▋        | 696/4176 [07:50<37:57,  1.53it/s]

...

{'loss': 55.4815, 'grad_norm': nan, 'learning_rate': 0.0, 'epoch': 5.74}


                                                   
 50%|█████     | 2089/4176 [23:36<22:48,  1.52it/s]

{'eval_loss': nan, 'eval_runtime': 6.4057, 'eval_samples_per_second': 217.462, 'eval_steps_per_second': 27.319, 'epoch': 6.0}


 51%|█████     | 2124/4176 [24:01<22:23,  1.53it/s]  

KeyboardInterrupt: 

In [ ]:
def predict(text):
    model.eval()
    text = "split: " + clean_text(text)
    
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs,
        max_length=96,
        num_beams=4
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
print(predict("रामोऽस्ति"))


In [ ]:
def evaluate(dataset):
    model.eval()
    correct = 0
    
    for inp, gold in dataset:
        pred = predict(inp.replace("split: ", ""))
        if pred.strip() == gold.strip():
            correct += 1
    
    return correct / len(dataset)

accuracy = evaluate(test_data)
print("Exact Match Accuracy:", accuracy)
